# Prepare chess datasets

Run all cells in order to **clean the raw data, inspect the report, then create
train / validation / test splits**. Adjust the settings below before running.

```text
dataset/raw/ → cleaning → dataset/processed/cleaned.txt → splitting → dataset/splits/
```

| Stage | Outputs |
| --- | --- |
| Clean | `cleaned.txt`, `provenance.jsonl`, `report.json` in `dataset/processed/` |
| Split | `train.txt`, `validation.txt`, `test.txt`, `split_info.json` in `dataset/splits/` |

Raw sources remain unchanged. Reruns replace these disposable preparation outputs;
they do not change the frozen `dataset/references/chess-v1/` dataset used by
reference experiments. See the [dataset guide](../docs/datasets.md) for the rules
and reference preparation commands. The preparation logic uses only the Python
standard library.


## 1. Setup and settings

Open this notebook from the repository or `notebooks/` directory. The seed and
ratios control the split. Balancing is off by default: enabling it repeats
minority-class rows **in training only**. Validation and test retain their
original class distribution. Stratification preserves class proportions; it
does not make classes equally frequent.


In [ ]:
from pathlib import Path
import sys

project_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'scripts' / 'clean_dataset.py').is_file()
                     and (path / 'scripts' / 'split_dataset.py').is_file()
                     and (path / 'dataset').is_dir()), None)
if project_root is None:
    raise FileNotFoundError('Open this notebook from the MyTorch project or notebooks/ folder')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from scripts.clean_dataset import clean_dataset
from scripts.split_dataset import split_dataset

seed = 42
ratios = (0.8, 0.1, 0.1)  # train, validation, test
balance_training = False

dataset_dir = project_root / 'dataset'
source_dir = dataset_dir / 'raw' if (dataset_dir / 'raw').is_dir() else dataset_dir
input_file = dataset_dir / 'processed' / 'cleaned.txt'
output_dir = dataset_dir / 'splits'
print('Raw sources:', source_dir)
print('Cleaned data:', input_file)
print('Splits:', output_dir)
print(f'Seed: {seed} | Ratios: {ratios} | Balance training: {balance_training}')


## 2. Clean raw sources

Read every `.txt` source recursively. Validate FENs and labels, normalize class
labels, remove duplicates, and exclude conflicting groups. A conflict means
identical model inputs have different class labels; every record in that group
is excluded. The cleaner checks format, not chess legality or label correctness.


In [ ]:
report = clean_dataset(dataset_dir)
print('Cleaned data:', dataset_dir / 'processed' / 'cleaned.txt')
print('Report:', dataset_dir / 'processed' / 'report.json')


In [ ]:
print(f"Records: {report['records']:,} | Retained: {report['accepted']:,} | "
      f"Duplicates: {report['duplicates']:,} | Rejected: {len(report['rejected']):,} | "
      f"Conflicting records: {report['conflicting_records']:,}")

largest = max(report['by_class'].values(), default=1) or 1
for label, count in report['by_class'].items():
    bar = '█' * round(30 * count / largest)
    print(f'{label:<10} {count:>10,}  {bar}')


### Sources and issues

`Retained` counts distinct output positions originating from each source. A
position found in two source files appears in both counts. The full references
for rejected and conflicting records are retained in `report.json`; the output
below shows at most 20 of each.


In [ ]:
print(f"{'Source':<44} {'Records':>10} {'Retained':>10} {'Rejected':>10}")
for source, counts in report['by_source'].items():
    print(f"{source:<44} {counts['records']:>10,} {counts['retained']:>10,} {counts['rejected']:>10,}")

for item in report['rejected'][:20]:
    print(f"Rejected {item['file']}:{item['line']}: {item['reason']}")
for group in report['conflicts'][:20]:
    print('Conflict:', ', '.join(f"{ref['file']}:{ref['line']} ({ref['label']})"
                                 for ref in group['sources']))
if len(report['rejected']) > 20 or len(report['conflicts']) > 20:
    print('Additional details: dataset/processed/report.json')


## 3. Split the cleaned data

Use the freshly generated `cleaned.txt`. Group model-visible inputs, shuffle
within each class with the selected seed, and assign groups to train, validation,
and test. The splitter checks coverage and absence of overlap before writing.
Each class needs at least three distinct groups to occur in all three splits.


In [ ]:
info = split_dataset(input_file, output_dir, seed=seed, ratios=ratios,
                     balance_training=balance_training)
print('Saved to:', output_dir)
print('Input SHA-256:', info['input_sha256'])


### Split counts and integrity

`Before` counts each original row once. `Output` includes optional repetitions
in training. With identical sources and settings, rerunning the notebook
reproduces the same files and hashes.


In [ ]:
print(f"{'Split':<12} {'Stage':<8} {'Nothing':>10} {'Check':>10} {'Checkmate':>10} {'Total':>10}")
for name in ('train', 'validation', 'test'):
    for stage, section in (('Before', 'before_balancing'), ('Output', 'output')):
        counts = info[section][name]
        print(f"{name:<12} {stage:<8} {counts['Nothing']:>10,} {counts['Check']:>10,} "
              f"{counts['Checkmate']:>10,} {sum(counts.values()):>10,}")
print('Overlapping model inputs:', info['overlapping_inputs'])


## 4. Use the prepared data

| File | Use |
| --- | --- |
| `dataset/splits/train.txt` | Update model weights |
| `dataset/splits/validation.txt` | Select settings and the best checkpoint; pass it through `--validation` |
| `dataset/splits/test.txt` | Evaluate the final selected model |
| `dataset/splits/split_info.json` | Inspect the input hash, settings, class counts, and overlap check |

For comparable reference experiments, use the **frozen reference paths** from
the [dataset guide](../docs/datasets.md#4-frozen-reference-datasets), rather than
these disposable outputs. Continue with the [training guide](../docs/training.md)
and [evaluation workflow](../docs/evaluation.md).
